# Derma Hybrid Model (Kaggle Ready)

This notebook runs end-to-end training/evaluation for `derma-hybrid-model` on HAM10000 in Kaggle.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/derma-hybrid-model')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

requirements_path = PROJECT_ROOT / 'requirements.txt'
if requirements_path.exists():
    print(f'Installing dependencies from {requirements_path}')
    get_ipython().system(f'pip install -q -r {requirements_path}')
else:
    print('No requirements.txt found; skipping install step.')

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from derma_hybrid_model import (
    DermoscopyDataset,
    HybridDermClassifier,
    MulticlassFocalLoss,
    build_transforms,
    compute_class_weights,
    create_train_val_test_split,
    evaluate_model,
    save_checkpoint,
    train_one_epoch,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Kaggle dataset path (update DATASET_ROOT if your dataset slug differs).
DATASET_ROOT = Path('/kaggle/input/skin-cancer-mnist-ham10000')
METADATA_PATH = DATASET_ROOT / 'HAM10000_metadata.csv'

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f'Metadata not found at {METADATA_PATH}. Update DATASET_ROOT to your HAM10000 dataset location.'
    )

image_part_1 = DATASET_ROOT / 'HAM10000_images_part_1'
image_part_2 = DATASET_ROOT / 'HAM10000_images_part_2'
single_image_dir = DATASET_ROOT / 'HAM10000_images'

MERGED_IMAGE_DIR = Path('/kaggle/working/HAM10000_images_merged')
MERGED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

if single_image_dir.exists():
    IMAGE_ROOT = single_image_dir
else:
    for directory in (image_part_1, image_part_2):
        if directory.exists():
            for image_file in directory.glob('*.jpg'):
                target = MERGED_IMAGE_DIR / image_file.name
                if not target.exists():
                    os.symlink(image_file, target)
    IMAGE_ROOT = MERGED_IMAGE_DIR

metadata = pd.read_csv(METADATA_PATH)
print('Metadata rows:', len(metadata))
print('Image root:', IMAGE_ROOT)

In [ ]:
# Training configuration
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 1
LEARNING_RATE = 1e-4
ACCUMULATION_STEPS = 2
USE_AMP = True
USE_PRETRAINED_BACKBONES = False  # Set True if pretrained weights are available in your Kaggle runtime.

In [ ]:
splits = create_train_val_test_split(metadata, seed=SEED)
base_transform, minority_transform = build_transforms(image_size=IMAGE_SIZE)

train_dataset = DermoscopyDataset(
    splits['train'],
    image_root=IMAGE_ROOT,
    transform=base_transform,
    minority_transform=minority_transform,
)
val_dataset = DermoscopyDataset(
    splits['val'],
    image_root=IMAGE_ROOT,
    transform=base_transform,
)
test_dataset = DermoscopyDataset(
    splits['test'],
    image_root=IMAGE_ROOT,
    transform=base_transform,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print({k: len(v) for k, v in splits.items()})

In [ ]:
model = HybridDermClassifier(pretrained=USE_PRETRAINED_BACKBONES).to(device)
class_weights = compute_class_weights(splits['train']).to(device)
criterion = MulticlassFocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device=device,
        accumulation_steps=ACCUMULATION_STEPS,
        use_amp=USE_AMP,
    )
    val_metrics = evaluate_model(model, val_loader, device=device)
    print(
        f'Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | '
        f'macro_f1={val_metrics["macro_f1"]:.4f} | balanced_acc={val_metrics["balanced_accuracy"]:.4f}'
    )

In [ ]:
test_metrics = evaluate_model(model, test_loader, device=device)
print('Test macro_f1:', test_metrics['macro_f1'])
print('Test balanced_accuracy:', test_metrics['balanced_accuracy'])

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/derma_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = save_checkpoint(OUTPUT_DIR, epoch=EPOCHS, model=model, optimizer=optimizer)
metrics_path = OUTPUT_DIR / 'test_metrics.json'

import json
with metrics_path.open('w') as f:
    json.dump(test_metrics, f, indent=2)

print('Saved checkpoint:', checkpoint_path)
print('Saved metrics:', metrics_path)